# Example: Human Protein Atlas — Subcellular Phenotypes

This notebook demonstrates how to analyze large-scale subcellular protein localization patterns using the Human Protein Atlas (HPA) dataset.

**Dataset Information:**
- **Source:** [Human Protein Atlas (HPA)](https://www.proteinatlas.org/)
- **Description:** High-resolution microscopy images of human proteins in various cell lines.
- **Metadata:** Provided in a master FoV CSV file mapping paths to protein locations and cell types.

**Workflow Overview:**
1. Load HPA metadata and extract image filenames.
2. Discover image files recursively.
3. Extract DINOv2 embeddings (optimized for high resolution).
4. Compute basic morphological and intensity properties.
5. Find prototypical images for each cell type.
6. Interactively explore subcellular localization patterns.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Specify the paths to your HPA data and metadata CSV.

In [ ]:
DATA_DIR = "/datasets/HPA/512_whole_images"
METADATA_PATH = "/datasets/HPA/Master_fovHPA_512.csv"

## 2. Load Metadata & Processing

We load the metadata and prepare a filename column for matching.

In [ ]:
import os

df_metadata = pd.read_csv(METADATA_PATH)

# Select relevant columns and extract filename
df_metadata = df_metadata[["Path", "protein_location", "cell_type"]]
df_metadata["filename"] = df_metadata["Path"].apply(lambda x: os.path.splitext(os.path.basename(x))[0])

print(f"Loaded {len(df_metadata)} HPA entries.")
df_metadata.head()

## 3. Data Discovery

Discover the whole images and link them with the metadata.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    str(DATA_DIR), 
    metadata_fn=make_dataframe_metadata_fn(df_metadata)
)
file_df.head()

## 4. Feature Extraction

Process images to get embeddings and compute basic properties. We use a larger `resize_size` for HPA images.

In [ ]:
pheno.process_images(
    wrapper,
    batch_size=32,
    resize_size=518,
    channel_mode="split",
)

df_properties = pheno.compute_properties(
    property_preset="basic",
    n_jobs=8,
)

## 5. Embedding plot

Visualize the embeddings in a 2D plot.

In [ ]:
pheno.plot_tsne(color_by="cell_type")

## 6. Interactive Exploration

Explore the subcellular localization patterns.

In [ ]:
e = pheno.create_interactive_explorer(hover_features=["cell_type", "protein_location"])